# Анализ вакансий креативных индустрий в России

Notebook выполняет полный цикл:
1. парсинг Telegram-каналов;
2. объединение данных;
3. очистка и фильтрация вакансий;
4. классификация вакансий по профессиональным направлениям;
5. извлечение навыков;
6. формирование финального датасета для анализа.


## 1. Импорт библиотек и настройки

In [ ]:
import os
import re
from collections import Counter

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 250)

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

from dotenv import load_dotenv
from telethon import TelegramClient
from python_socks import ProxyType

RAW_DATA_PATH = os.path.join(DATA_DIR, "data_merged.csv")
CLEAN_VACANCIES_PATH = os.path.join(DATA_DIR, "data_clean.csv")
REMOVED_POSTS_PATH = os.path.join(DATA_DIR, "data_removed.csv")
FINAL_DATASET_PATH = os.path.join(DATA_DIR, "vacancies_dataset.csv")


## 2. Парсинг Telegram-каналов

Перед запуском парсинга нужно создать файл `.env` рядом с notebook'ом:

```text
API_ID=...
API_HASH=...
PHONE_NUMBER=...
PASSWORD=...
```

In [30]:
RUN_PARSER = False

CHANNELS = {
    "digital_jobster": "https://t.me/digital_jobster",
    "vdhl_good": "https://t.me/vdhl_good",
    "designhunters": "https://t.me/designhunters",
}

# Максимальное число сообщений с одного канала.
LIMIT = 100000

# Настройки прокси
USE_PROXY = True
PROXY_HOST = "127.0.0.1"
PROXY_PORT = 12334


In [ ]:
async def parse_channel(client, channel_name, channel_url, limit=LIMIT):
    """Парсит один Telegram-канал и возвращает список словарей с сообщениями."""
    print(f"Начинаю парсинг канала: {channel_name} ({channel_url})")

    messages_data = []

    try:
        entity = await client.get_entity(channel_url)

        async for message in client.iter_messages(entity, limit=limit):
            if message.text:
                messages_data.append({
                    "channel": channel_name,
                    "date": message.date.isoformat() if message.date else None,
                    "message_id": message.id,
                    "sender": message.sender_id,
                    "text": message.text.strip()
                })

        print(f"  -> Собрано текстовых сообщений: {len(messages_data)}")

    except Exception as error:
        print(f"  -> Ошибка при парсинге {channel_name}: {error}")

    return messages_data


async def run_parser():
    """Запускает парсинг всех каналов и сохраняет общий файл data_merged.csv."""

    load_dotenv()

    api_id = os.getenv("API_ID")
    api_hash = os.getenv("API_HASH")
    phone_number = os.getenv("PHONE_NUMBER")
    password = os.getenv("PASSWORD")

    if not api_id or not api_hash or not phone_number:
        raise ValueError(
            "Не найдены API_ID, API_HASH или PHONE_NUMBER. "
            "Проверь файл .env."
        )

    proxy = None

    if USE_PROXY:

        proxy = {
            "proxy_type": ProxyType.SOCKS5,
            "addr": PROXY_HOST,
            "port": PROXY_PORT,
            "rdns": True,
        }

    client = TelegramClient(
        "my_parser_session",
        int(api_id),
        api_hash,
        device_model="PC",
        system_version="Windows 10",
        app_version="4.15.2",
        lang_code="ru",
        system_lang_code="ru-RU",
        connection_retries=5,
        retry_delay=3,
        timeout=30,
        proxy=proxy,
    )

    await client.start(phone=phone_number, password=password)
    print("Клиент Telegram успешно запущен.")

    all_messages = []

    for channel_name, channel_url in CHANNELS.items():
        channel_messages = await parse_channel(client, channel_name, channel_url)
        all_messages.extend(channel_messages)

        channel_path = os.path.join(DATA_DIR, f"{channel_name}.csv")
        pd.DataFrame(channel_messages).to_csv(
            channel_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"  -> Файл сохранён: {channel_path}")

    await client.disconnect()

    df_raw = pd.DataFrame(all_messages)
    df_raw.to_csv(RAW_DATA_PATH, index=False, encoding="utf-8-sig")

    print("-" * 50)
    print(f"Итоговый файл сохранён: {RAW_DATA_PATH}")
    print(f"Всего сообщений: {len(df_raw)}")

    return df_raw


if RUN_PARSER:
    df_raw = await run_parser()
else:
    print("Парсинг пропущен")


Парсинг пропущен


## 3. Загрузка и базовая подготовка данных

In [32]:
def load_source_data(path=RAW_DATA_PATH):
    """Загружает объединённый файл с постами."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Файл {path} не найден. "
            "Сначала запусти парсер или положи data_merged.csv в папку data."
        )

    df = pd.read_csv(path)

    required_columns = ["channel", "date", "message_id", "sender", "text"]
    missing_columns = [column for column in required_columns if column not in df.columns]

    if missing_columns:
        raise ValueError(f"В датасете не хватает столбцов: {missing_columns}")

    return df


df = load_source_data()

print(f"Размер исходного датасета: {df.shape}")
display(df.head())


Размер исходного датасета: (30708, 5)


,channel,date,message_id,sender,text
0,digital_jobster,2026-06-15T11:14:01+00:00,5916,-1001448839765,"Резюме: **Marketing manager**\n\n__👩‍💻 Карташева Анна\nНик tg: ____@akartasheva__\n\n**Возраст:** 32 \n\n__Начинала в НКО, выросла до руководителя международных проектов. Потом решила, что хочу в маркетинг — и пошла. Внутри маркетинга прошла путь..."
1,digital_jobster,2026-06-15T09:44:01+00:00,5915,-1001448839765,"**SMM-менеджер в творческую студию Звезд\n\nРаботодатель:**\n[__Студия Звёзд__](https://star-studios.ru/)__ (творческая/образовательная студия для детей и подростков). Ищем SMM-менеджера, который умеет не просто «вести соцсети», а системно привод..."
2,digital_jobster,2026-06-15T08:02:01+00:00,5913,-1001448839765,Резюме: **Бренд-менеджер / Проджект-менеджер**\n\n__👩‍💻 Шабанова Алина\nНик tg: ____@Aspid_Suslikovich__\n\n**Возраст:** 27 \n**Локация:** Санкт-Петербург / Удаленно \n\n**▪️Какую работу ищете и чем хотите заниматься?**\nИщу постоянную работу бре...
3,digital_jobster,2026-06-12T11:14:01+00:00,5909,-1001448839765,"**Контент-менеджер на управление контент производством в AI-агентство\n\nРаботодатель:**\n__Мы — AI-контент агентство ICG. Создаем 10,000+ коротких видео в месяц. \n\nДелаем видео для брендов в формате говорящей головы для Reels / Shorts / TikTo..."
4,digital_jobster,2026-06-12T09:36:14+00:00,5908,-1001448839765,__Резюме:__ **Руководитель отдела PR и внешних коммуникаций**\n\n__👩‍💻 Шаврова Анна Валериевна\nНик tg: ____@Anna_Shavrova____\nНомер телефона: ____8-915-480-29-02__\n\n**Возраст:** 58 лет\n**Локация: **Москва\n\n__Эксперт в области стратегически...


In [ ]:
def normalize_channel_name(value):
    """Приводит названия каналов к единому виду."""
    value = str(value).strip()

    mapping = {
        "https://t.me/digital_jobster": "digital_jobster",
        "https://t.me/vdhl_good": "vdhl_good",
        "https://t.me/designhunters": "designhunters",
        "digital_jobster": "digital_jobster",
        "vdhl_good": "vdhl_good",
        "designhunters": "designhunters",
    }

    return mapping.get(value, value)


df["channel"] = df["channel"].apply(normalize_channel_name)
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

before_cleaning = len(df)

df = df[df["text"].str.len() > 0].copy()
df = df.drop_duplicates(subset=["channel", "message_id"]).copy()
df = df.drop_duplicates(subset=["channel", "text"]).copy()

after_cleaning = len(df)

print("БАЗОВАЯ ОЧИСТКА")
print("-" * 50)
print(f"Было строк: {before_cleaning}")
print(f"Осталось строк: {after_cleaning}")
print(f"Удалено пустых/дублирующихся строк: {before_cleaning - after_cleaning}")

print("\nКоличество постов по каналам:")
display(df["channel"].value_counts())


БАЗОВАЯ ОЧИСТКА
--------------------------------------------------
Было строк: 30708
Осталось строк: 28987
Удалено пустых/дублирующихся строк: 1721

Количество постов по каналам:


channel
vdhl_good          20458
digital_jobster     5006
designhunters       3523
Name: count, dtype: int64

## 4. Фильтрация вакансий через балльную систему

На этом этапе посты не удаляются по одному слову. Для каждого текста считается `vacancy_score`: положительные признаки повышают вероятность вакансии, отрицательные — понижают.

Отдельно выделены признаки резюме. Важно: само слово «резюме» не считается признаком резюме кандидата, потому что в вакансиях часто пишут «присылайте резюме». Поэтому резюме отсекаются только по контекстным признакам: `Резюме:`, `ищу работу`, `обо мне`, `возраст:`, `локация:` и т.п.


In [34]:
# Баллы за признаки вакансии.
POSITIVE_PATTERNS = {
    r"\bваканси[яи]\b": 3,
    r"\bищем\b": 3,
    r"\bищет\b": 3,
    r"\bтребуется\b": 3,

    r"\bнуж(?:ен|на|ны)\b": 2,
    r"\bв\s+команду\b": 2,

    r"\bобязанност[а-я]*\b": 3,
    r"\bтребовани[яй]\b": 3,
    r"\bуслови[яй]\b": 3,

    r"\bзадач[а-я]*\b": 2,
    r"\bчто\s+делать\b": 2,
    r"\bмы\s+предлагаем\b": 2,

    r"\bотклик[а-я]*\b": 2,
    r"\bотправляйте\b": 2,
    r"\bприсылайте\s+(?:свое\s+|своё\s+)?резюме\b": 2,
    r"\bжд[её]м\s+(?:свое\s+|своё\s+)?резюме\b": 2,

    r"\bзарплат[а-я]*\b|\bзп\b|\bоплат[а-я]*\b|\bоклад[а-я]*\b": 2,

    r"\bfull[- ]?time\b|\bpart[- ]?time\b|\bremote\b": 2,
    r"\bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b": 2,
}

# Баллы за признаки не-вакансий.
NEGATIVE_PATTERNS = {
    r"\bстать[яи]\b": -3,
    r"\bподборк[а-я]*\b": -3,
    r"\bдайджест[а-я]*\b": -3,
    r"\bинтервью\b": -3,

    r"\bвебинар[а-я]*\b": -3,
    r"\bкурс[а-я]*\b|\bобучени[а-я]*\b": -3,
    r"\bконференци[яи]\b": -2,
    r"\bмитап[а-я]*\b|\bлекци[яи]\b": -2,

    r"\bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b": -3,
    r"\bподписывайтесь\b": -2,
}

# Явные признаки резюме кандидата.
RESUME_PATTERNS = [
    r"(^|\n)\s*[_*\s]*резюме[_*\s]*\s*:",
    r"\bрезюме\s*:\s*[*_]*\s*[а-яa-z]",

    r"\bищу\s+работу\b",
    r"\bищу\s+постоянную\s+работу\b",
    r"\bищу\s+удал[её]нную\s+работу\b",
    r"\bищу\s+проект[а-я]*\b",

    r"\bоткрыт[а]?\s+к\s+предложениям\b",
    r"\bготов[а]?\s+к\s+сотрудничеству\b",

    r"\bобо\s+мне\b",
    r"\bмой\s+опыт\b",
    r"\bмои\s+навыки\b",
    r"\bмой\s+стек\b",

    r"\bвозраст\s*:",
    r"\bлокаци[яи]\s*:",
    r"\bгород\s+проживания\s*:",

    r"\bкак\s+кандидат\b",
]

VACANCY_THRESHOLD = 5


def find_regex_patterns(text, patterns):
    """Возвращает список регулярных выражений, найденных в тексте."""
    return [
        pattern
        for pattern in patterns
        if re.search(pattern, text)
    ]


def calculate_vacancy_score(text):
    """Считает балл вероятности того, что пост является вакансией."""
    text_lower = str(text).lower().replace("ё", "е")

    positive_found = find_regex_patterns(text_lower, POSITIVE_PATTERNS)
    negative_found = find_regex_patterns(text_lower, NEGATIVE_PATTERNS)
    resume_found = find_regex_patterns(text_lower, RESUME_PATTERNS)

    score = sum(POSITIVE_PATTERNS[pattern] for pattern in positive_found)
    score += sum(NEGATIVE_PATTERNS[pattern] for pattern in negative_found)

    return pd.Series({
        "vacancy_score": score,
        "positive_signals_count": len(positive_found),
        "negative_signals_count": len(negative_found),
        "resume_signals_count": len(resume_found),
        "positive_signals": ", ".join(positive_found),
        "negative_signals": ", ".join(negative_found),
        "resume_signals": ", ".join(resume_found),
        "has_resume_signal": len(resume_found) > 0,
    })


score_columns = df["text"].apply(calculate_vacancy_score)
df = pd.concat([df, score_columns], axis=1)

df["is_vacancy"] = (
    (df["vacancy_score"] >= VACANCY_THRESHOLD)
    & (~df["has_resume_signal"])
)

df_vacancies = df[df["is_vacancy"]].copy()
df_removed = df[~df["is_vacancy"]].copy()

df_vacancies.to_csv(CLEAN_VACANCIES_PATH, index=False, encoding="utf-8-sig")
df_removed.to_csv(REMOVED_POSTS_PATH, index=False, encoding="utf-8-sig")

print("СТАТИСТИКА ФИЛЬТРАЦИИ")
print("-" * 50)
print(f"Всего постов после базовой очистки: {len(df)}")
print(f"Оставлено как вакансии: {len(df_vacancies)}")
print(f"Отсеяно как не-вакансии: {len(df_removed)}")
print(f"Доля оставленных вакансий: {len(df_vacancies) / len(df):.2%}")
print(f"Доля отсеянных постов: {len(df_removed) / len(df):.2%}")
print(f"Отсеяно по контекстным признакам резюме: {df['has_resume_signal'].sum()}")

channel_stats = (
    df.groupby("channel")
      .agg(
          total_posts=("text", "count"),
          vacancies=("is_vacancy", "sum"),
          avg_score=("vacancy_score", "mean"),
          resume_posts=("has_resume_signal", "sum"),
      )
)

channel_stats["removed"] = channel_stats["total_posts"] - channel_stats["vacancies"]
channel_stats["vacancy_share"] = channel_stats["vacancies"] / channel_stats["total_posts"]

display(channel_stats)

print("\nПРИМЕРЫ ОСТАВЛЕННЫХ ПОСТОВ С БАЛЛОМ 5–7")
print("-" * 50)

suspicious_kept = df[
    (df["is_vacancy"])
    & (df["vacancy_score"].between(5, 7))
]

display(
    suspicious_kept[
        [
            "channel", "date", "vacancy_score",
            "positive_signals_count", "negative_signals_count",
            "resume_signals_count", "positive_signals",
            "negative_signals", "resume_signals", "text"
        ]
    ].sample(min(30, len(suspicious_kept)), random_state=42)
)

print("\nПРИМЕРЫ ОТСЕЯННЫХ РЕЗЮМЕ")
print("-" * 50)

removed_resume = df_removed[df_removed["has_resume_signal"]]

display(
    removed_resume[
        [
            "channel", "date", "vacancy_score",
            "positive_signals", "negative_signals",
            "resume_signals", "text"
        ]
    ].sample(min(30, len(removed_resume)), random_state=42)
)


СТАТИСТИКА ФИЛЬТРАЦИИ
--------------------------------------------------
Всего постов после базовой очистки: 28987
Оставлено как вакансии: 16962
Отсеяно как не-вакансии: 12025
Доля оставленных вакансий: 58.52%
Доля отсеянных постов: 41.48%
Отсеяно по контекстным признакам резюме: 364


,total_posts,vacancies,avg_score,resume_posts,removed,vacancy_share
channel,,,,,,
designhunters,3523,2354,6.899517,40,1169,0.668181
digital_jobster,5006,2655,5.800240,232,2351,0.530364
vdhl_good,20458,11953,5.862205,92,8505,0.584270



ПРИМЕРЫ ОСТАВЛЕННЫХ ПОСТОВ С БАЛЛОМ 5–7
--------------------------------------------------


,channel,date,vacancy_score,positive_signals_count,negative_signals_count,resume_signals_count,positive_signals,negative_signals,resume_signals,text
13747,vdhl_good,2023-04-11 15:28:39+00:00,5,3,1,0,"\bнуж(?:ен|на|ны)\b, \bобязанност[а-я]*\b, \bуслови[яй]\b",\bкурс[а-я]*\b|\bобучени[а-я]*\b,,"**ВЕДУЩИЙ МАСТЕР-КЛАССОВ по керамике в Гончарную студию №1.\n\n**Крупнейшая сеть гончарных студий в России. Мы расширяем нашу команду мастеров и в поиске сотрудника, который будет любить дело, как мы, и не оставаться безразличным к результату раб..."
16084,vdhl_good,2022-05-31 11:03:58+00:00,6,3,1,0,"\bобязанност[а-я]*\b, \bтребовани[яй]\b, \bуслови[яй]\b",\bкурс[а-я]*\b|\bобучени[а-я]*\b,,"**МЛАДШИЙ МОУШН-ДИЗАЙНЕР в РБК.\n\n**Обязанности:\n— Изготовление моушн-дизайна: телевизионного дизайна, графического дизайна, инфографики, заставок, титров, графики для размещения в цифровой среде (сайт, социальные сети, мобильные устройства и т..."
20341,vdhl_good,2021-01-13 17:46:43+00:00,6,3,0,0,"\bв\s+команду\b, \bзадач[а-я]*\b, \bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b",,,"МЕНЕДЖЕР/КООРДИНАТОР в команду IvlevGroup.\n\nПроекты разные, интересные онлайн и офлайн!\n\nОсновная задача — это организация и управление Проектом в команде:\n• участие в разработке стратегии и продвижения проекта;\n• участие в разработке и соз..."
15583,vdhl_good,2022-08-08 12:55:50+00:00,5,3,1,0,"\bищем\b, \bтребовани[яй]\b, \bзадач[а-я]*\b",\bкурс[а-я]*\b|\bобучени[а-я]*\b,,"**РАБОТА В НКО.\n\nСОЦИАЛЬНЫЙ ПОМОЩНИК в «Партнёрство каждому ребенку», г. Санкт-Петербург.\n\n**Особенный ребенок требует заботы и внимания 24/7. Программа «Передышка» c 2008 года помогает семьям детей с инвалидностью, позволяя родителям получит..."
1191,digital_jobster,2024-03-29 14:18:10+00:00,6,4,1,0,"\bтребовани[яй]\b, \bотклик[а-я]*\b, \bзарплат[а-я]*\b|\bзп\b|\bоплат[а-я]*\b|\bоклад[а-я]*\b, \bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b",\bстать[яи]\b,,"#Job_вакансия #маркетинг #копирайтер #seo\n\n**SEO-копирайтер **\n\n**Работодатель:**\nHonest-team\n\nПривет 👋,\nМы - молодая digital-команда , с адекватным руководством без бюрократии и микроменеджмента.\n\n**Что нужно делать:**\n • Заниматься S..."
27374,designhunters,2026-04-25 06:04:27+00:00,6,4,1,0,"\bищем\b, \bотклик[а-я]*\b, \bзарплат[а-я]*\b|\bзп\b|\bоплат[а-я]*\b|\bоклад[а-я]*\b, \bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b",\bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b,,"**Коммуникационный дизайнер** в Звук\n\nИщем смелого дизайнера, который поможет развивать визуальный стиль бренда и не боится экспериментировать.\n\n**Чем нужно будет заниматься:**\n— Развитие визуальной айдентики\n— Создание дизайн-концепций и р..."
21050,vdhl_good,2020-08-31 05:58:31+00:00,5,3,1,0,"\bищем\b, \bнуж(?:ен|на|ны)\b, \bобязанност[а-я]*\b",\bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b,,"The Breakfast растет, и нам нужен водительский и любопытный человек с отличными навыками написания, производства и управления всяким контентом для нашей рассылки Instagram и электронной почты. \n \nОфициальная часть: \nЗавтрак - это мобильное при..."
12421,vdhl_good,2023-08-25 11:34:40+00:00,5,3,1,0,"\bищем\b, \bнуж(?:ен|на|ны)\b, \bуслови[яй]\b",\bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b,,"**PR-МЕНЕДЖЕР в PR-агентство «About You».\n\n**Мы — креативное пиар агентство “About You”, которое работает с селебрити (блогерами, артистами), бизнесами и экспертами.\nИщем в нашу команду человека с амбициями и любовью к работе.\n\nНаши проекты ..."
11535,vdhl_good,2023-11-29 07:01:23+00:00,5,3,1,0,"\bтребовани[яй]\b, \bуслови[яй]\b, \bзадач[а-я]*\b",\bкурс[а-я]*\b|\bобучени[а-я]*\b,,"**ИЛЛЮСТРАТОР в Билайн.\n\nТребования:**\nОпыт работы не менее 3 лет на аналогичной позиции.\nПрофессиональное владение Photoshop, Illustrator.\nНаличие интересного портфолио.\nЧувство стиля, знание трендов.\nГотовность выполнить тестовое задание..."
2696,digital_jobster,2021-04-04 12


ПРИМЕРЫ ОТСЕЯННЫХ РЕЗЮМЕ
--------------------------------------------------


,channel,date,vacancy_score,positive_signals,negative_signals,resume_signals,text
4156,digital_jobster,2020-05-06 14:30:00+00:00,-6,,"\bкурс[а-я]*\b|\bобучени[а-я]*\b, \bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b",\bмой\s+опыт\b,"#резюме #предлагаю_услуги #помогу\n\n**SMM специалист\n\n**Меня зовут Екатерина, я СММ-специалист. Сейчас успешно прохожу курс от действующего профессионала natalieviner. Могу отправить кейсы её команды.\n\n**Мой опыт работы:\n**1,5 месяца.\n\n**..."
97,digital_jobster,2026-05-15 09:12:01+00:00,1,"\bзадач[а-я]*\b, \bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b",\bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b,"(^|\n)\s*[_*\s]*резюме[_*\s]*\s*:, \bлокаци[яи]\s*:",__Резюме:__ **Продуктовый digital-маркетолог B2B / IT**\n\n__👩‍💻 Наталья Гражданкина\nНик tg: ____@Natalia_grazh____\nWhatsApp: ____+7__-__925__-__875-65-93__\n\n**Локация:** Таиланд / удалённо\n\n▪️**Какую работу ищете и чем хотите заниматься?**...
39,digital_jobster,2026-06-02 12:26:08+00:00,12,"\bваканси[яи]\b, \bищем\b, \bотклик[а-я]*\b, \bзарплат[а-я]*\b|\bзп\b|\bоплат[а-я]*\b|\bоклад[а-я]*\b, \bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b",,\bлокаци[яи]\s*:,"**PR-менеджер в маркетинговое агентство About you agency\n\nРаботодатель:**\n__В экспертный отдел ____aboutyou.agency____ ищем PR-менеджера.\n\nИщем человека, который хорошо ориентируется в медиа-поле, понимает индустрию, следит за инфоповесткой ..."
29668,designhunters,2020-08-21 10:10:07+00:00,7,"\bуслови[яй]\b, \bмы\s+предлагаем\b, \bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b",,\bлокаци[яи]\s*:,"**Дизайнер UI/UX | от 100.000 рублей на руки**\n\nУровень кандидата: Senior\nКомпания: rhino-digital.com\nЛокация: Москва \nФорма и режим работы: Удаленная работа, полный рабочий день.\n\nКто мы?\nМолодая амбициозная компания, которая занимается ..."
242,digital_jobster,2026-03-13 11:14:01+00:00,-1,\bудаленн[а-я]*\b|\bудал[её]нн[а-я]*\b|\bгибрид[а-я]*\b|\bофис[а-я]*\b,\bреклам[а-я]*\b|\bпартнерск[а-я]*\b|\bпартн[её]рск[а-я]*\b,"(^|\n)\s*[_*\s]*резюме[_*\s]*\s*:, \bвозраст\s*:, \bлокаци[яи]\s*:",__Резюме:__ **Marketing Project Manager / Growth Manager**\n\n__👩‍💻 Дарья Романова\nНик tg: ____@darleeeyaaa____\nНомер телефона: ____8-993-955-38-02__\n\n**Возраст:** __20__\n**Локация:** __удалённо / Москва (гибрид)__\n\n**▪️Какую работу ищу и ...
4095,digital_jobster,2020-05-15 18:00:00+00:00,-3,,\bкурс[а-я]*\b|\bобучени[а-я]*\b,\bобо\s+мне\b,#резюме #предлагаю_услуги #помогу\n\n**SMM - специалист / копирайтер\n\nОбо мне:\n**📝Патологическая влюблённость в жизнь и все ее составляющие. \n📝Чрезмерная пытливость. Способна залезть под кожу и найти там что-то интересненькое. \n📝Обостренное ...
324,digital_jobster,2026-02-10 10:39:13+00:00,4,"\bзадач[а-я]*\b, \bзарплат[а-я]*\b|\bзп\b|\bоплат[а-я]*\b|\bоклад[а-я]*\b",,"(^|\n)\s*[_*\s]*резюме[_*\s]*\s*:, \bлокаци[яи]\s*:",__Резюме:__ **Маркетолог-стратег / Chief Marketing Officer**\n\n__👩‍💻Леоненко Юлия\nНик tg: ____@leonenkoyulia____\nНомер телефона: ____+7(929)842-00-98__\n**Локация:** Краснодар\n\n**▪️Какую работу ищу чем хочу заниматься:**\n• Разработка и внед...
2756,digital_jobster,2021-03-14 13:30:01+00:00,8,"\bваканси[яи]\b, \bтребуется\b, \bнуж(?:ен|на|ны)\b",,\bобо\s+мне\b,"#вакансия #требуется #ищу \n\n**Веб-дизайнер\n**\nВсем привет!\n\nМеня зовут Галия, эксперт- психотерапевт про здоровый эгоизм и любовь к себе.\n\nИщу веб-дизайнера с услугами копирайта для создания одностраничного сайта-визитки со списком услуг...."
3839,digital_jobster,2020-06-25 06:30:02+00:00,0,,,\bищу\s+проект[а-я]*\b,"#резюме #предлагаю_услуги #помогу\n\n**Создание сайтов\n\n**Меня зовут Таня. Я работаю в сфере SMM уже 4 года.\nНа данный момент занимаюсь созданием сайтов на платформе Tilda, в связи с чем ищу проекты для портфолио.\n\n**Для Вас:\n**— Создание с..."
3065,digital_jobster,2020-12-12 06:30:00+00:00,-4,\bзадач[а-я]*\b,"\bкурс[а-я]*\b|\bобучени[а-я]*\b, \bреклам

In [35]:
print("Примеры отсеянных постов:")
display(
    df_removed[[
        "channel", "date", "vacancy_score",
        "positive_signals_count", "negative_signals_count", "text"
    ]].sample(min(10, len(df_removed)), random_state=42)
)

print("Примеры оставленных вакансий:")
display(
    df_vacancies[[
        "channel", "date", "vacancy_score",
        "positive_signals_count", "negative_signals_count", "text"
    ]].sample(min(10, len(df_vacancies)), random_state=42)
)


Примеры отсеянных постов:


,channel,date,vacancy_score,positive_signals_count,negative_signals_count,text
13155,vdhl_good,2023-06-19 06:57:19+00:00,0,0,0,"Начало рабочей недели. Самое время написать сильное резюме и разместить его в нашем телеграм-канале: https://t.me/vdhl_resume.\n\nПЛЮСЫ КАНАЛА:\n👍3250 подписчиков, 25% из которых работодатели\n👍Каждое резюме просматривают от 500 до 1000 раз\n👍Нек..."
26690,vdhl_good,2018-04-13 07:14:18+00:00,-3,0,1,SMM-МЕНЕДЖЕР в рекламное агентство Qmarketing.\nПодробности и контакты: http://vdhl.ru/smm-menedzher-110
26722,vdhl_good,2018-04-10 10:35:37+00:00,0,0,0,АССИСТЕНТ в SMM-отдел KIOSKO.\n\nПодробности и контакты: http://vdhl.ru/assistent-v-smm-otdel-2
23551,vdhl_good,2019-09-17 08:05:19+00:00,0,0,0,"РЕЖИССЕР МОНТАЖА в продакшн Mastiff.\n\nНеобходимые умения, чтобы попасть в нашу команду:\n— Монтаж телепрограмм в программе Adobe Premiere Pro.\n— Желательно владеть программой After Effects\n— Сбор программы с мультикамеры. \n— Навыки клипового..."
20423,vdhl_good,2020-12-22 08:36:50+00:00,3,1,0,"Старший PR-МЕНЕДЖЕР в агентство Main PR.\n\nСпециализация агентства: корпоративные и внутренние коммуникации, Digital & SMM, личное позиционирование, медиа-коучинг, брендинг и дизайн.\n\nОбязанности:\n— придумываем и детально прорабатываем способ..."
17188,vdhl_good,2021-12-07 05:21:25+00:00,0,0,0,"#ЦИТАТА_ДНЯ_VDHL\n\nМы не в силах изменить нашу действительность до тех пор, пока не станем задавать более подходящих вопросов. Томас Леонард"
1959,digital_jobster,2021-11-25 11:01:00+00:00,-4,1,2,"#резюме #предлагаю_услугу #помогу \n\n**Продюсер\n\n**Меня зовут Надежда, я Инфопродюсер.\n\n**Предлагаю следующие услуги:\n**Консультация-5000 рублей \n\n**В чем заключается консультация:\n**— Мы проводим с вами созвон, где я подробно рассказыва..."
5161,digital_jobster,2019-10-17 13:23:02+00:00,-4,2,3,#резюме #помогу #предлагаю_услуги\n\n**Менеджер блогера** \n\nДобрый день!\nЯ Ксюша работаю менеджером -блогеров.\n\n**Умею** :\nВ мои услуги входит:\n- разбор директа\n- набор на рекламу\n- организация именного гива\n- поиск рекламы для продви...
19591,vdhl_good,2021-04-02 05:22:20+00:00,0,0,0,"#ЦИТАТА_ДНЯ_VDHL\n\nЖизнь — это то, что с тобой происходит, пока ты строишь планы. Джон Леннон"
26779,vdhl_good,2018-04-03 17:28:14+00:00,0,0,0,ТЕЛЕВЕДУЩИЙ на телерадиоканал Страна FM.\n\nПодробности и контакты: http://vdhl.ru/televedushhij-2


Примеры оставленных вакансий:


,channel,date,vacancy_score,positive_signals_count,negative_signals_count,text
29068,designhunters,2022-05-30 18:00:21+00:00,9,3,0,"[​​](https://telegra.ph/file/6d5a9deb19867f20fa1aa.png)ZION ищет дизайнера лендингов\n \nZION — агентство международного интернет-маркетинга. Работаем с 2016 года. Мы работаем с комплексным маркетингом в России и за рубежом (есть кейсы в Европе, ..."
2726,digital_jobster,2021-03-25 07:21:00+00:00,21,8,0,"#вакансия #требуется #ищу \n\n**Копирайтер (Копирайт/рерайт)\n\nОбязанности:\n**— Поиск и написание постов (рерайт) по двум блогам на яндекс.дзене на выбор. \n\n**Тематика одного: **точные науки, математика, программирование, IT, алгоритмы, блокч..."
27146,designhunters,2026-06-15 06:02:19+00:00,9,4,0,"UX-редактор, Т-Банк \n\nИщем редактора, который будет прорабатывать логику и тексты интерфейсов в приложении Т-Банка\n\nЗадачи: \n- продумывать логику фичей вместе с дизайнером и продактом\n- писать UX-тексты: онбординги, заголовки, кнопки, алерт..."
5257,vdhl_good,2026-06-02 15:42:01+00:00,6,2,0,"**SMM-МЕНЕДЖЕР в международную сеть развлечений Big Fun Family.**\n\nBig Fun Family — международная сеть уникальных развлекательных музеев (Россия, Испания, США) имеет прочные позиции в индустрии развлечений на международном рынке. Первый комплек..."
16396,vdhl_good,2022-04-06 10:30:37+00:00,14,5,0,"В анимационную компанию ЯРКО требуется БРЕНД-МЕНЕДЖЕР. \n\nОбязанности: \nРазработка стратегии развития анимационных проектов; \nФормирование процесса бренд девелопмента (аналитика, продуктовое видение, концепция, позиционирование, бренд айдентик..."
8936,vdhl_good,2024-10-16 12:21:01+00:00,6,6,3,"**В команду игровой образовательной лаборатории **[**Lilelo Games **](https://lilelo.games/#we)**нужен ПРОДЮСЕР/ПРОДЖЕКТ-МЕНЕДЖЕР с опытом работы в онлайн-школах (удалёнка).** \n \n**С нас:** яркий, необычный и актуальный продукт в сфере детского..."
14413,vdhl_good,2023-01-27 12:08:16+00:00,7,5,2,**РЕДАКТОР ОТДЕЛА МОДЫ журналов Marie Claire и Psychologies.** \n\n**Обязанности: \n**— работа с рекламным отделом по получению информации о договоренностях с партнерами по объемам размещения и объемам и формату поддержек; \n— планирование поддер...
2864,digital_jobster,2021-02-10 13:01:15+00:00,16,7,1,"#вакансия #требуется #ищу \n\n**Project Таргетолог\n\nОбязанности:\n**— Создание стратегии проекта на этапе продажи (Какие услуги продаём, что предлагаем и тд.).\n— Подбор команды под проект (Штатных и фрилансеров).\n— Контроль показателей реклам..."
8355,vdhl_good,2025-01-20 08:58:19+00:00,7,4,1,**Бренд одежды AZI Land ищет МАРКЕТОЛОГА в Москве (преимущественно удаленная работа)\n\nЗадачи:\n**> Формирование предложений для развития бренда и маркетинговых стратегий\n> Мониторинг и анализ конкурентной среды для выявления тенденций и возмож...
25270,vdhl_good,2018-12-27 13:19:55+00:00,8,3,0,"КОРРЕСПОНДЕНТ редакции политической информации в ТАСС. \n \nОбязанности: \nМониторинг событий и новостей в области внутриполитической информации. \nОперативная работа с новостным материалом, подготовка новостных сообщений. \nУстановление и поддер..."


## 5. Классификация вакансий по специальностям

In [36]:
df = df_vacancies.copy()

# Словарь профессиональных направлений.
PROFESSION_PATTERNS = {
    "Дизайн": [
        r"\bux\b", r"\bui\b", r"\bux\s*/?\s*ui\b",
        r"\bproduct\s+designer\b",
        r"\bпродуктов[а-я]*\s+дизайн[её]р[а-я]*\b",
        r"\bдизайн[её]р[а-я-]*\b",
        r"\bграфическ[а-я]*\s+дизайн[её]р[а-я]*\b",
        r"\bgraphic\s+designer\b",
        r"\bweb\s+designer\b",
        r"\bвеб[-\s]?дизайн[её]р[а-я]*\b",
        r"\bбренд[-\s]?дизайн[её]р[а-я]*\b",
        r"\bbrand\s+designer\b",
        r"\bmotion\s+designer\b",
        r"\bмоушн[-\s]?дизайн[её]р[а-я]*\b",
        r"\bиллюстратор[а-я]*\b",
        r"\billustrator\b",
        r"\bаниматор[а-я]*\b",
        r"\b[23]d\s+animator\b",
        r"\b[23]d\s+аниматор[а-я]*\b",
        r"\b[23]d\s+artist\b",
        r"\bartist\b",
        r"\bхудожник[а-я-]*\b",
        r"\bарт[-\s]?директор[а-я]*\b",
        r"\bart\s+director\b",
        r"\bвизуализатор[а-я]*\b",
        r"\bсторисмейкер[а-я]*\b",
        r"\bstorymaker\b",
        r"\bfigma\b",
    ],

    "Маркетинг / SMM / PR": [
        r"\bsmm\b", r"\bсмм\b",
        r"\bмаркетолог[а-я]*\b",
        r"\bмаркетинг[а-я]*\b",
        r"\bmarketing\b",
        r"\bpr\b", r"\bпиар[а-я]*\b",
        r"\bтаргетолог[а-я]*\b",
        r"\bтаргет[а-я]*\b",
        r"\bperformance\b",
        r"\bperformance\s+marketing\b",
        r"\bcrm\b",
        r"\bcrm\s+manager\b",
        r"\bбренд[-\s]?менеджер[а-я]*\b",
        r"\bbrand\s+manager\b",
        r"\bкомьюнити[а-я-]*\b",
        r"\bcommunity\s+manager\b",
        r"\bdigital\s+marketing\b",
        r"\binfluenc[ea]r?\b",
        r"\bинфлюенс[а-я]*\b",
        r"\bблогер[а-я]*\b",
        r"\bменеджер[а-я]*\s+по\s+работе\s+с\s+блогер[а-я]*\b",
        r"\bконтент[-\s]?маркетолог[а-я]*\b",
        r"\bemail[-\s]?маркетолог[а-я]*\b",
        r"\bemail\s+marketing\b",
        r"\bseo\b",
    ],

    "Продюсирование / видео / production": [
        r"\bпродюсер[а-я]*\b",
        r"\bproducer\b",
        r"\bproduction\b",
        r"\bпродакшн[а-я]*\b",
        r"\bвидеограф[а-я]*\b",
        r"\bмонтаж[её]р[а-я]*\b",
        r"\bоператор[а-я]*\b",
        r"\bрежисс[её]р[а-я]*\b",
        r"\bmotion\b",
        r"\breels\b",
        r"\bрилс[а-я]*\b",
        r"\bвидео[а-я-]*\b",
        r"\byoutube\b",
        r"\bютуб[а-я]*\b",
        r"\bвидеомонтаж[а-я]*\b",
        r"\bмонтаж[а-я]*\b",
        r"\bагент[а-я]*\s+режисс[её]р[а-я]*\b",
        r"\bагент[а-я]*\s+оператор[а-я]*\b",
    ],

    "Project / product / account management": [
        r"\bproject\s+manager\b",
        r"\bпроджект[а-я-]*\b",
        r"\bпроектн[а-я]*\s+менеджер[а-я]*\b",
        r"\bменеджер[а-я]*\s+проект[а-я]*\b",
        r"\bproduct\s+manager\b",
        r"\bпродакт[а-я-]*\b",
        r"\bпродуктов[а-я]*\s+менеджер[а-я]*\b",
        r"\bменеджер[а-я]*\s+продукт[а-я]*\b",
        r"\baccount\s+manager\b",
        r"\bаккаунт[а-я-]*\b",
        r"\bclient\s+manager\b",
        r"\bменеджер[а-я]*\s+по\s+клиент[а-я]*\b",
        r"\bменеджер[а-я]*\s+по\s+работе\s+с\s+клиент[а-я]*\b",
        r"\bменеджер[а-я]*\s+по\s+работе\s+с\s+партн[её]р[а-я]*\b",
        r"\bменеджер[а-я]*\s+по\s+развити[а-я]*\b",
        r"\bменеджер[а-я]*\s+отдел[а-я]*\b",
        r"\bменеджер[а-я]*\s+направлени[а-я]*\b",
        r"\bevent\s+manager\b",
        r"\bevent\s+director\b",
        r"\bevent[-\s]?директор[а-я]*\b",
        r"\bивент[а-я-]*\b",
        r"\bevent\b",
        r"\bкуратор[а-я]*\b",
        r"\bкоординатор[а-я]*\b",
        r"\bфандрайзер[а-я]*\b",
        r"\bfundraiser\b",
        r"\bменеджер[а-я]*\s+по\s+закупк[а-я]*\b",
        r"\bзакупк[а-я]*\b",
        r"\bарт[-\s]?менеджер[а-я]*\b",
        r"\bgallery\s+manager\b",
    ],

    "Контент / редактура / копирайтинг": [
        r"\bкопирайтер[а-я]*\b",
        r"\bcopywriter\b",
        r"\bредактор[а-я]*\b",
        r"\beditor\b",
        r"\bконтент[а-я-]*\b",
        r"\bcontent\b",
        r"\bавтор[а-я]*\b",
        r"\bжурналист[а-я]*\b",
        r"\bкорреспондент[а-я]*\b",
        r"\bкорректор[а-я]*\b",
        r"\bрайтер[а-я]*\b",
        r"\bwriter\b",
        r"\bсценарист[а-я]*\b",
        r"\bобозревател[а-я]*\b",
        r"\bведущ[а-я]*\b",
        r"\bмодератор[а-я]*\b",
        r"\bmoderator\b",
        r"\bконтент[-\s]?менеджер[а-я]*\b",
        r"\bcontent\s+manager\b",
        r"\bшеф[-\s]?редактор[а-я]*\b",
        r"\bглавн[а-я]*\s+редактор[а-я]*\b",
        r"\bстарш[а-я]*\s+редактор[а-я]*\b",
    ],

    "IT / аналитика": [
        r"\bаналитик[а-я]*\b",
        r"\banalyst\b",
        r"\bdata\b",
        r"\bdata\s+analyst\b",
        r"\bsql\b",
        r"\bpython\b",
        r"\bразработчик[а-я]*\b",
        r"\bdeveloper\b",
        r"\bfrontend\b",
        r"\bbackend\b",
        r"\bqa\b",
        r"\bтестировщик[а-я]*\b",
        r"\bbi\b",
        r"\bpower\s+bi\b",
        r"\btableau\b",
        r"\bпрограммист[а-я]*\b",
        r"\bno[-\s]?code\b",
        r"\bnocode\b",
        r"\bсистемн[а-я]*\s+аналитик[а-я]*\b",
        r"\bбизнес[-\s]?аналитик[а-я]*\b",
    ],

    "HR / рекрутинг": [
        r"\bhr\b",
        r"\bэйчар[а-я]*\b",
        r"\bрекрутер[а-я]*\b",
        r"\brecruiter\b",
        r"\btalent\s+acquisition\b",
        r"\bhr\s+manager\b",
        r"\bкадровик[а-я]*\b",
        r"\bpeople\s+partner\b",
        r"\bспециалист[а-я]*\s+по\s+подбор[а-я]*\b",
        r"\bменеджер[а-я]*\s+по\s+подбор[а-я]*\b",
    ],

    "Продажи / business development": [
        r"\bsales\b",
        r"\bпродаж[а-я]*\b",
        r"\bменеджер[а-я]*\s+по\s+продаж[а-я]*\b",
        r"\bbusiness\s+development\b",
        r"\bbizdev\b",
        r"\bbdm\b",
        r"\bкоммерческ[а-я]*\s+менеджер[а-я]*\b",
        r"\bsales\s+manager\b",
    ],

    "Администрирование / ассистенты": [
        r"\bассистент[а-я]*\b",
        r"\bпомощник[а-я]*\b",
        r"\bадминистратор[а-я]*\b",
        r"\bадминистративн[а-я]*\s+менеджер[а-я]*\b",
        r"\bсекретар[а-я]*\b",
        r"\bофис[-\s]?менеджер[а-я]*\b",
        r"\bличн[а-я]*\s+ассистент[а-я]*\b",
        r"\bперсональн[а-я]*\s+ассистент[а-я]*\b",
        r"\bбэк[-\s]?офис[а-я]*\b",
        r"\bback\s+office\b",
        r"\bадминистратор[а-я]*\s+сайт[а-я]*\b",
    ],
}

PROFESSION_PRIORITY = list(PROFESSION_PATTERNS.keys())


In [37]:
def normalize_text(text):
    """Нормализует текст для регулярных выражений и поиска."""
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"&nbsp;", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_first_lines(text, n=5):
    """Возвращает первые n строк поста, где обычно находится название вакансии."""
    return " ".join(str(text).split("\n")[:n])


def count_pattern_matches(text, patterns):
    """Считает количество найденных паттернов и возвращает список совпадений."""
    found_patterns = [
        pattern
        for pattern in patterns
        if re.search(pattern, text)
    ]
    return len(found_patterns), found_patterns


def classify_profession(text):
    """Классифицирует вакансию по профессиональному направлению."""
    full_text = normalize_text(text)
    title_part = normalize_text(get_first_lines(text, n=5))

    scores = {}
    found = {}

    for profession, patterns in PROFESSION_PATTERNS.items():
        title_count, title_found = count_pattern_matches(title_part, patterns)
        full_count, full_found = count_pattern_matches(full_text, patterns)

        # Первые строки важнее, так как там чаще всего находится название вакансии.
        score = title_count * 3 + full_count

        scores[profession] = score
        found[profession] = sorted(set(title_found + full_found))

    max_score = max(scores.values())

    if max_score == 0:
        return pd.Series({
            "profession": "Другое",
            "profession_score": 0,
            "profession_keywords_found": "",
        })

    best_professions = [
        profession
        for profession, score in scores.items()
        if score == max_score
    ]

    for profession in PROFESSION_PRIORITY:
        if profession in best_professions:
            best_profession = profession
            break

    return pd.Series({
        "profession": best_profession,
        "profession_score": max_score,
        "profession_keywords_found": ", ".join(found[best_profession]),
    })


profession_columns = df["text"].apply(classify_profession)
df = pd.concat([df, profession_columns], axis=1)

profession_stats = (
    df["profession"]
    .value_counts(normalize=True)
    .reset_index()
)

profession_stats.columns = ["profession", "share"]

print("РАСПРЕДЕЛЕНИЕ ВАКАНСИЙ ПО СПЕЦИАЛЬНОСТЯМ")
display(profession_stats)

print("ПРИМЕРЫ ИЗ КАТЕГОРИИ 'ДРУГОЕ'")
other_examples = df[df["profession"] == "Другое"][
    ["channel", "profession_score", "text"]
].sample(
    min(20, (df["profession"] == "Другое").sum()),
    random_state=42,
)

display(other_examples)


РАСПРЕДЕЛЕНИЕ ВАКАНСИЙ ПО СПЕЦИАЛЬНОСТЯМ


,profession,share
0,Маркетинг / SMM / PR,0.223912
1,Контент / редактура / копирайтинг,0.218901
2,Дизайн,0.208348
3,Продюсирование / видео / production,0.113548
4,Project / product / account management,0.067445
5,Другое,0.062728
6,Администрирование / ассистенты,0.057246
7,Продажи / business development,0.031541
8,IT / аналитика,0.008784
9,HR / рекрутинг,0.007546


ПРИМЕРЫ ИЗ КАТЕГОРИИ 'ДРУГОЕ'


,channel,profession_score,text
2353,digital_jobster,0,"#вакансия #требуется #ищу \n\n**Финансист\n\n**Команда Академии для Женщин в поисках позитивного, ответственного человека, который любит работать с цифрами, постоянно обучаться новому в своей профессии и следить за порядком в делах.\n\n**Обязанно..."
30032,designhunters,0,**Как легко найти удалённую работу\n\n**Канал [Фрилансеры](https://t.me/seejobfl) - это самые интересные вакансии для более 50 онлайн-профессий.\n\nУдаленная работа - это часто 2-3 часа занятости в день и стабильная зарплата от 30 000 до 250 000 ...
11613,vdhl_good,0,"**МЕНЕДЖЕР ПОДДЕРЖКИ ПУТЕШЕСТВЕННИКОВ в Tripster.**\n\nТрипстер — место, где каждое путешествие превращается в настоящее приключение, а каждая экскурсия в уникальный опыт.\n\n**Что необходимо делать?**\n– Помогать путешественникам и гидам пользов..."
18229,vdhl_good,0,Мультимедиа Арт Музей ищет СПЕЦИАЛИСТА по учету музейных предметов. \n\nОбязанности: \n— учет и хранение музейных предметов.\n \nУсловия: \n— график 5/2; \n— зарплата по итогам собеседования.\n \nТребования: \n— высшее образование; \n— опыт работ...
19336,vdhl_good,0,"Командорский заповедник открыл набор ВОЛОНТЁРОВ на лето. \n\nКомандорские острова – это уникальное место, объединившее мир живой природы представителей двух континентов Евразии и Северной Америки и дикую, первозданную красоту островных ландшафтов..."
14430,vdhl_good,0,"**РУКОВОДИТЕЛИ ВНЕБЮДЖЕТНОЙ СТУДИИ в новый и современный «Культурный центр «Строгино».\n\nОбязанности: \n**- проведение групповых занятий; \n- организация мастер-классов, открытых уроков, участие в мероприятиях Культурного центра; \n- ведение жур..."
14564,vdhl_good,0,"**РАБОТА В НКО.\n\nРУКОВОДИТЕЛЬ ПРОГРАММЫ работы с интернатами в Детский хоспис «Дом с маяком».\n\nОбязанности:\n**— налаживание взаимодействия с интернатами;\n— координация деятельности хосписа в отношении детей, находящихся в интернатах;\n— орг..."
14783,vdhl_good,0,**МЕНЕДЖЕР-ТИМЛИД проектов государственной важности в студию Артемия Лебедева.\n\n**Вакансия открыта до 20 декабря.\n\nНужный нам человек чувствует себя уверенно при ведении масштабных проектов с государственными и окологосударственными компаниям...
5209,vdhl_good,0,**НОЧНАЯ ДЕЖУРНАЯ/НОЧНОЙ ДЕЖУРНЫЙ в лагерь («Проект ГРАНИ»).**\n\n[**«Проект ГРАНИ»**](https://grani.pro/)** **– это альтернатива лагерю в Подмосковье для детей и подростков от 6 до 17 лет.\n\n**Вакантные даты: 15-26 июня ❗️**\nВакансию можно сов...
11595,vdhl_good,0,"**СЕЙЛЗ-МЕНЕДЖЕР в SWIFT Studio (удалёнка).**\n\n**Кто мы такие?**\nВ SWIFT мы занимаемся созданием анимации для са-а-амых разных задач. Мы делаем имиджевые ролики, информационные, продающие, просто красивые, динамичные или плавные — едва ли не н..."


## 6. Извлечение навыков из текстов вакансий

Навыки извлекаются по контролируемому словарю regex-паттернов. Такой подход выбран вместо автоматического извлечения существительных, потому что он даёт более интерпретируемые признаки для статистического анализа и защиты проекта.


In [38]:
# Словарь навыков и инструментов.
SKILL_PATTERNS = {
    # Дизайн и UX/UI
    "Figma": [r"\bfigma\b"],
    "Photoshop": [r"\bphotoshop\b", r"\badobe\s+photoshop\b"],
    "Illustrator": [r"\billustrator\b", r"\badobe\s+illustrator\b"],
    "InDesign": [r"\bindesign\b", r"\badobe\s+indesign\b"],
    "Adobe XD": [r"\badobe\s+xd\b"],
    "Sketch": [r"\bsketch\b"],
    "Tilda": [r"\btilda\b", r"\bтильд[а-я]*\b"],
    "Webflow": [r"\bwebflow\b"],
    "Readymag": [r"\breadymag\b"],
    "Framer": [r"\bframer\b"],
    "Canva": [r"\bcanva\b", r"\bканв[а-я]*\b"],
    "ProtoPie": [r"\bprotopie\b"],
    "Principle": [r"\bprinciple\b"],
    "User Research": [
        r"\buser\s+research\b",
        r"\bисследовани[а-я]*\s+пользовател[а-я]*\b",
    ],

    # Motion и видео
    "After Effects": [r"\bafter\s+effects\b"],
    "Premiere Pro": [r"\bpremiere\s+pro\b"],
    "DaVinci Resolve": [r"\bdavinci\s+resolve\b", r"\bdavinci\b"],
    "Final Cut Pro": [r"\bfinal\s+cut\b", r"\bfinal\s+cut\s+pro\b"],
    "CapCut": [r"\bcapcut\b"],
    "Blender": [r"\bblender\b"],
    "Cinema 4D": [r"\bcinema\s*4d\b", r"\bc4d\b"],

    # Маркетинг
    "SEO": [r"\bseo\b"],
    "Email Marketing": [
        r"\bemail\s+marketing\b",
        r"\bemail[-\s]?маркетинг[а-я]*\b",
    ],
    "CRM": [r"\bcrm\b"],
    "Google Analytics": [r"\bgoogle\s+analytics\b", r"\bga4\b"],
    "Яндекс Метрика": [r"\bяндекс\s+метрик[а-я]*\b", r"\bметрик[а-я]*\b"],
    "Google Ads": [r"\bgoogle\s+ads\b", r"\badwords\b"],
    "Яндекс Директ": [r"\bяндекс\s+директ\b"],
    "VK Ads": [r"\bvk\s+ads\b"],
    "Telegram Ads": [r"\btelegram\s+ads\b"],
    "Google Tag Manager": [r"\bgoogle\s+tag\s+manager\b", r"\bgtm\b"],

    # Аналитика
    "Excel": [r"\bexcel\b", r"\bэксел[а-я]*\b"],
    "SQL": [r"\bsql\b"],
    "Python": [r"\bpython\b"],
    "Power BI": [r"\bpower\s+bi\b"],
    "Tableau": [r"\btableau\b"],

    # Менеджмент
    "Notion": [r"\bnotion\b"],
    "Miro": [r"\bmiro\b"],
    "Jira": [r"\bjira\b"],
    "Trello": [r"\btrello\b"],
    "Asana": [r"\basana\b"],
    "Confluence": [r"\bconfluence\b"],
    "Monday": [r"\bmonday\b"],

    # CRM и продажи
    "Bitrix24": [r"\bbitrix24\b", r"\bбитрикс24\b"],
    "amoCRM": [r"\bamocrm\b"],

    # Разработка
    "HTML": [r"\bhtml\b"],
    "CSS": [r"\bcss\b"],
    "JavaScript": [r"\bjavascript\b"],
    "React": [r"\breact\b"],

    # AI
    "ChatGPT": [r"\bchatgpt\b", r"\bchat\s*gpt\b"],
    "GPT": [r"\bgpt[-\s]?4\b", r"\bgpt4\b", r"\bgpt\b"],
    "Claude": [r"\bclaude\b"],
    "Midjourney": [r"\bmidjourney\b"],
    "Stable Diffusion": [r"\bstable\s+diffusion\b"],
    "DALL-E": [r"\bdall[-\s]?e\b", r"\bdalle\b"],
    "Sora": [r"\bsora\b"],
}


In [39]:
def extract_skills(text):
    """Извлекает навыки из текста вакансии по словарю regex-паттернов."""
    text_norm = normalize_text(text)

    found_skills = []

    for skill, patterns in SKILL_PATTERNS.items():
        if any(re.search(pattern, text_norm) for pattern in patterns):
            found_skills.append(skill)

    return sorted(set(found_skills))


df["skills"] = df["text"].apply(extract_skills)
df["skills_count"] = df["skills"].apply(len)
df["skills_str"] = df["skills"].apply(lambda skills: ", ".join(skills))
df["has_skill"] = df["skills_count"] > 0

AI_SKILLS = {"ChatGPT", "GPT", "Claude", "Midjourney", "Stable Diffusion", "DALL-E", "Sora"}

df["has_ai_skill"] = df["skills"].apply(
    lambda skills: any(skill in AI_SKILLS for skill in skills)
)

skill_counter = Counter()

for skills in df["skills"]:
    skill_counter.update(skills)

top_skills = pd.DataFrame(
    skill_counter.most_common(30),
    columns=["skill", "count"],
)

top_skills["share"] = top_skills["count"] / len(df)

print("ТОП-30 НАВЫКОВ")
display(top_skills)

print("ОПИСАТЕЛЬНАЯ СТАТИСТИКА ПО КОЛИЧЕСТВУ НАВЫКОВ")
display(df["skills_count"].describe())

print(f"Доля вакансий с хотя бы одним найденным навыком: {df['has_skill'].mean():.2%}")
print(f"Доля вакансий с AI-навыками: {df['has_ai_skill'].mean():.2%}")


ТОП-30 НАВЫКОВ


,skill,count,share
0,Photoshop,1502,0.088551
1,Figma,1337,0.078823
2,Illustrator,952,0.056125
3,Яндекс Метрика,846,0.049876
4,CRM,743,0.043804
5,Excel,710,0.041858
6,After Effects,547,0.032249
7,SEO,378,0.022285
8,InDesign,378,0.022285
9,Tilda,321,0.018925


ОПИСАТЕЛЬНАЯ СТАТИСТИКА ПО КОЛИЧЕСТВУ НАВЫКОВ


count    16962.000000
mean         0.620210
std          1.188139
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          9.000000
Name: skills_count, dtype: float64

Доля вакансий с хотя бы одним найденным навыком: 31.15%
Доля вакансий с AI-навыками: 0.99%


## 7. Дополнительные признаки: формат работы

In [40]:
REMOTE_PATTERNS = [
    r"\bудаленн[а-я]*\b",
    r"\bудал[её]нн[а-я]*\b",
    r"\bудаленк[а-я]*\b",
    r"\bудал[её]нк[а-я]*\b",
    r"\bremote\b",
    r"\bиз\s+любой\s+точки\b",
    r"\bдистанционн[а-я]*\b",
]

HYBRID_PATTERNS = [
    r"\bгибрид[а-я]*\b",
    r"\bhybrid\b",
    r"\bчастично\s+удаленн[а-я]*\b",
    r"\bчастично\s+удал[её]нн[а-я]*\b",
]

OFFICE_PATTERNS = [
    r"\bофис[а-я]*\b",
    r"\bв\s+офисе\b",
    r"\boffice\b",
    r"\bочно\b",
    r"\bонлайн\s+не\s+рассматриваем\b",
]


def contains_any_regex(text, patterns):
    """Проверяет, найден ли хотя бы один regex-паттерн в тексте."""
    text_norm = normalize_text(text)
    return any(re.search(pattern, text_norm) for pattern in patterns)


df["is_remote"] = df["text"].apply(lambda text: contains_any_regex(text, REMOTE_PATTERNS))
df["is_hybrid"] = df["text"].apply(lambda text: contains_any_regex(text, HYBRID_PATTERNS))
df["is_office"] = df["text"].apply(lambda text: contains_any_regex(text, OFFICE_PATTERNS))

print("ФОРМАТ РАБОТЫ")
print("-" * 50)
print(f"Удалёнка: {df['is_remote'].mean():.2%}")
print(f"Гибрид: {df['is_hybrid'].mean():.2%}")
print(f"Офис: {df['is_office'].mean():.2%}")


ФОРМАТ РАБОТЫ
--------------------------------------------------
Удалёнка: 31.71%
Гибрид: 6.37%
Офис: 30.01%


## 8. Финальный датасет и сохранение результатов

In [41]:
final_columns = [
    "channel",
    "date",
    "message_id",
    "sender",
    "text",
    "vacancy_score",
    "is_vacancy",
    "profession",
    "profession_score",
    "profession_keywords_found",
    "skills",
    "skills_str",
    "skills_count",
    "has_skill",
    "has_ai_skill",
    "is_remote",
    "is_hybrid",
    "is_office",
]

df_final = df[final_columns].copy()

df_final.to_csv(FINAL_DATASET_PATH, index=False, encoding="utf-8-sig")

print("ФИНАЛЬНЫЙ ДАТАСЕТ")
print("-" * 50)
print(f"Размер: {df_final.shape}")
print(f"Файл сохранён: {FINAL_DATASET_PATH}")

display(df_final.head())


ФИНАЛЬНЫЙ ДАТАСЕТ
--------------------------------------------------
Размер: (16962, 18)
Файл сохранён: data\vacancies_dataset.csv


,channel,date,message_id,sender,text,vacancy_score,is_vacancy,profession,profession_score,profession_keywords_found,skills,skills_str,skills_count,has_skill,has_ai_skill,is_remote,is_hybrid,is_office
1,digital_jobster,2026-06-15 09:44:01+00:00,5915,-1001448839765,"**SMM-менеджер в творческую студию Звезд\n\nРаботодатель:**\n[__Студия Звёзд__](https://star-studios.ru/)__ (творческая/образовательная студия для детей и подростков). Ищем SMM-менеджера, который умеет не просто «вести соцсети», а системно привод...",9,True,Маркетинг / SMM / PR,6,"\bsmm\b, \bкомьюнити[а-я-]*\b, \bтаргет[а-я]*\b","[Excel, Яндекс Метрика]","Excel, Яндекс Метрика",2,True,False,True,False,False
3,digital_jobster,2026-06-12 11:14:01+00:00,5909,-1001448839765,"**Контент-менеджер на управление контент производством в AI-агентство\n\nРаботодатель:**\n__Мы — AI-контент агентство ICG. Создаем 10,000+ коротких видео в месяц. \n\nДелаем видео для брендов в формате говорящей головы для Reels / Shorts / TikTo...",10,True,Продюсирование / видео / production,8,"\breels\b, \bвидео[а-я-]*\b, \bмонтаж[а-я]*\b, \bмонтаж[её]р[а-я]*\b, \bпродакшн[а-я]*\b",[],,0,False,False,True,False,False
5,digital_jobster,2026-06-12 07:53:27+00:00,5907,-1001448839765,"**Контент-маркетолог в онлайн-школу творчества\n\nРаботодатель:**\n__Школа __[__миниатюрной игрушки__](https://miniatureschool.ru/miniature-course)__ работает более 7 лет.\n\nБольшое сообщество, \nонлайн-курсы, мастер-классы и подписка\nЭксперт р...",14,True,Маркетинг / SMM / PR,9,"\bконтент[-\s]?маркетолог[а-я]*\b, \bмаркетинг[а-я]*\b, \bмаркетолог[а-я]*\b",[],,0,False,False,True,False,False
6,digital_jobster,2026-06-11 16:09:01+00:00,5906,-1001448839765,"**Менеджер отдела продаж / Будущий тимлид (ОГЭ/ЕГЭ) в онлайн школу Базис**\n**\nРаботодатель:**\n__Онлайн-школа ""БАЗИС"" ищет сильного менеджера по продажам, который умеет продавать через звонки и чаты, понимает специфику рынка ОГЭ/ЕГЭ и хочет выр...",14,True,Продажи / business development,8,"\bменеджер[а-я]*\s+по\s+продаж[а-я]*\b, \bпродаж[а-я]*\b",[CRM],CRM,1,True,False,True,False,False
7,digital_jobster,2026-06-11 15:09:01+00:00,5905,-1001448839765,"**Adult Directories & Partnerships Manager\n\nРаботодатель:**\n__Международный product-стартап. Наши продукты- Telegram Mini Apps и веб платформы, где пользователи общаются с AI-персонажами (девушками), смотрят эксклюзивные материалы, дарят подар...",10,True,Маркетинг / SMM / PR,1,\bseo\b,"[Notion, SEO, Яндекс Метрика]","Notion, SEO, Яндекс Метрика",3,True,False,True,False,False


In [42]:
print("КРАТКОЕ РЕЗЮМЕ ПОДГОТОВКИ ДАННЫХ")
print("-" * 50)
print(f"Исходных постов после базовой очистки: {len(df_vacancies) + len(df_removed)}")
print(f"Вакансий после фильтрации: {len(df_final)}")
print(f"Порог vacancy_score: {VACANCY_THRESHOLD}")
print(f"Доля вакансий с найденными навыками: {df_final['has_skill'].mean():.2%}")
print(f"Доля вакансий с AI-навыками: {df_final['has_ai_skill'].mean():.2%}")

print("\nРаспределение профессий:")
display(df_final["profession"].value_counts(normalize=True).to_frame("share"))

print("\nТоп навыков:")
display(top_skills.head(15))


КРАТКОЕ РЕЗЮМЕ ПОДГОТОВКИ ДАННЫХ
--------------------------------------------------
Исходных постов после базовой очистки: 28987
Вакансий после фильтрации: 16962
Порог vacancy_score: 5
Доля вакансий с найденными навыками: 31.15%
Доля вакансий с AI-навыками: 0.99%

Распределение профессий:


,share
profession,
Маркетинг / SMM / PR,0.223912
Контент / редактура / копирайтинг,0.218901
Дизайн,0.208348
Продюсирование / видео / production,0.113548
Project / product / account management,0.067445
Другое,0.062728
Администрирование / ассистенты,0.057246
Продажи / business development,0.031541
IT / аналитика,0.008784



Топ навыков:


,skill,count,share
0,Photoshop,1502,0.088551
1,Figma,1337,0.078823
2,Illustrator,952,0.056125
3,Яндекс Метрика,846,0.049876
4,CRM,743,0.043804
5,Excel,710,0.041858
6,After Effects,547,0.032249
7,SEO,378,0.022285
8,InDesign,378,0.022285
9,Tilda,321,0.018925
